# Prompt Engineering for Text-to-SQL
## Iterating on system prompt, few-shot examples, and schema context

In [ ]:
from llm.client import generate_sql
from llm.prompts import build_prompt, SYSTEM_PROMPT, FEW_SHOTS
from llm.schema_context import get_schema_context
from guardrails.sql_validator import validator
from guardrails.parser import parse_sql
from database.connection import engine
from sqlalchemy import text

In [ ]:
# Test questions covering different SQL patterns
test_questions = [
    "Show all products in the Electronics category",
    "What's the total revenue by country?",
    "Find products priced above the average",
    "Monthly order count for 2024",
    "Top 5 customers by lifetime value with their favorite category",
]

In [ ]:
def evaluate_question(question: str, show_context: bool = False):
    schema = get_schema_context()
    prompt = build_prompt(question, schema)
    
    if show_context:
        print("=== SCHEMA CONTEXT ===")
        print(schema[:500] + "...")
        print()
    
    print(f"Q: {question}")
    generation = generate_sql(prompt)
    print(f"SQL: {generation.sql}")
    
    validation = validator.validate(generation.sql)
    print(f"Valid: {validation.valid}")
    if validation.errors:
        print(f"Errors: {validation.errors}")
    if validation.warnings:
        print(f"Warnings: {validation.warnings}")
    
    if validation.valid:
        with engine.connect() as conn:
            result = conn.execute(text(generation.sql))
            rows = result.fetchmany(5)
            print(f"Results: {rows}")
    print()

In [ ]:
# Run evaluation
for q in test_questions:
    evaluate_question(q)

In [ ]:
# Experiment: modify system prompt and compare
print("Current system prompt:")
print(SYSTEM_PROMPT)
print()
print(f"Few-shot count: {len(FEW_SHOTS)}")
print()
for i, shot in enumerate(FEW_SHOTS, 1):
    print(f"{i}. {shot['question']} -> {shot['sql'][:60]}...")